In [1]:
import cv2
import numpy as np
import os, glob
from pathlib import Path
import pandas as pd

In [2]:
BASE_DIR   = r"/home/hasan/coding/MoneyLens/ai/Dataset_ocr"
PC_DIR  = os.path.join(BASE_DIR, "Preprocessing Citra", "dataset-grayscaling")
OUT_DIR    = os.path.join(BASE_DIR, "preprocessed")

SPLITS = ["train", "valid", "test"]

IMG_H = 32
IMG_W = 128
CHANNELS = 1
MIN_CROP = 5

W_SCALE = 1.05
H_SCALE = 1.10

CLASS_MAP = {
    0: "QTY",
    1: "harga_satuan",
    2: "nama_produk",
    3: "tanggal",
    4: "total_harga_barang",
    5: "total_transaksi",
}

In [3]:
def read_yolo_labels(label_path, img_w, img_h):
    labels = []

    if not os.path.exists(label_path):
        return labels

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls_id = int(parts[0])
            cx = float(parts[1]) * img_w
            cy = float(parts[2]) * img_h
            w  = float(parts[3]) * img_w
            h  = float(parts[4]) * img_h

            # scale ringan
            w *= W_SCALE
            h *= H_SCALE

            x1 = int(cx - w/2)
            y1 = int(cy - h/2)
            x2 = int(cx + w/2)
            y2 = int(cy + h/2)

            labels.append({
                "class_id": cls_id,
                "class_name": CLASS_MAP.get(cls_id),
                "x1": max(0, x1),
                "y1": max(0, y1),
                "x2": min(img_w, x2),
                "y2": min(img_h, y2),
            })

    return labels

In [4]:
def crop_with_padding(img, x1, y1, x2, y2):
    ih, iw = img.shape[:2]

    bw = x2 - x1
    bh = y2 - y1

    pad_x = max(3, int(bw * 0.05))
    pad_y = max(3, int(bh * 0.08))

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(iw, x2 + pad_x)
    y2 = min(ih, y2 + pad_y)

    return img[y1:y2, x1:x2]

In [5]:
def preprocess_crop(crop):
    h, w = crop.shape[:2]

    if h < MIN_CROP or w < MIN_CROP:
        raise ValueError("Crop terlalu kecil")

    if len(crop.shape) == 3:
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    resized = cv2.resize(crop, (IMG_W, IMG_H))
    normalized = resized.astype(np.float32) / 255.0

    return normalized.reshape(IMG_H, IMG_W, CHANNELS)

In [6]:
def process_image(img_path, label_path, crops_dir, arrays_dir):

    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Gagal load {img_path}")

    ih, iw = img.shape[:2]
    stem = Path(img_path).stem

    labels = read_yolo_labels(label_path, iw, ih)

    # DEBUG
    debug = img.copy()
    for lbl in labels:
        cv2.rectangle(debug, (lbl["x1"], lbl["y1"]), (lbl["x2"], lbl["y2"]), (0,255,0), 2)
        cv2.putText(debug, lbl["class_name"], (lbl["x1"], lbl["y1"]-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    cv2.imwrite(os.path.join(crops_dir, f"{stem}_DEBUG.jpg"), debug)

    counts = {}

    for lbl in labels:
        cls = lbl["class_name"]
        idx = counts.get(cls, 0)
        counts[cls] = idx + 1

        try:
            crop = crop_with_padding(img, lbl["x1"], lbl["y1"], lbl["x2"], lbl["y2"])

            # save raw
            raw_dir = os.path.join(crops_dir, "raw")
            os.makedirs(raw_dir, exist_ok=True)
            cv2.imwrite(os.path.join(raw_dir, f"{stem}_{cls}_{idx}.png"), crop)

            arr = preprocess_crop(crop)

            preview = (arr[:,:,0] * 255).astype(np.uint8)
            cv2.imwrite(os.path.join(crops_dir, f"{stem}_{cls}_{idx}.png"), preview)

            np.save(os.path.join(arrays_dir, f"{stem}_{cls}_{idx}.npy"), arr)

        except Exception as e:
            print(f"[ERROR] {cls}: {e}")

In [7]:
for split in SPLITS:

    img_dir = os.path.join(PC_DIR, split, "images")
    lbl_dir = os.path.join(PC_DIR, split, "labels")

    crops_dir = os.path.join(OUT_DIR, split, "crops")
    arrays_dir = os.path.join(OUT_DIR, split, "arrays")

    os.makedirs(crops_dir, exist_ok=True)
    os.makedirs(arrays_dir, exist_ok=True)

    img_paths = sorted(
        glob.glob(os.path.join(img_dir, "*.jpg")) +
        glob.glob(os.path.join(img_dir, "*.png"))
    )

    print(f"\n[{split}] {len(img_paths)} gambar")

    for i, img_path in enumerate(img_paths, 1):
        stem = Path(img_path).stem
        label_path = os.path.join(lbl_dir, stem + ".txt")

        try:
            process_image(img_path, label_path, crops_dir, arrays_dir)
            print(f"[{i}] OK {stem}")
        except Exception as e:
            print(f"[{i}] ERROR {stem} → {e}")


[train] 282 gambar
[1] OK 10_4_26-nota-tinta-isolasi__jpg.rf.9ccf27b8b344e1d5cb1ae9b63477e853
[2] OK 10_4_26-nota-trashbag__jpg.rf.45eca1cbce4cd88ad99d7e41cb9f9b4b
[3] OK 16_03_26-nota-hvs_jpg.rf.0025fea66265e3cd3b8dd5bd88cc9c43
[4] OK 16_4_26-nota-trashbag__jpg.rf.aa0788dfb9362567b7ed8fffff4f7743
[5] OK 20260507_155458_jpg.rf.0b96da501b12f92d9497b22d4abeb758
[6] OK Dipindai_20260508-1744-pdf_page_1_png.rf.b4b5e0a91f713d662317a25d05b9e033
[7] OK IMG-20260222-WA0097_jpg.rf.a18c5a4b5b3bd3776fad4d6687761439
[8] OK IMG-20260326-WA0066_jpg.rf.0362a7f9daaa5489f9604a7905cf9a3b
[9] OK IMG-20260329-WA0043_jpg.rf.92562d8b223a6897dd826560f8738c2c
[10] OK IMG-20260416-WA0048_jpg.rf.5472753104596f6d63ca85ae3294a476
[11] OK IMG-20260416-WA0103_jpg.rf.4876da76daa690ac9f0c2a0d2c856172
[12] OK IMG-20260418-WA0133_jpg.rf.eed8890acdba77e2c15f7cfc0a143cb4
[13] OK IMG-20260508-WA0174_jpg.rf.5152c842a5e1b49efc172af90536beea
[14] OK IMG-20260508-WA0176_jpg.rf.b31c03ad97596be2c1f4b6eddf22146a
[15] OK IMG2026